# 1. Vectorless RAG — Complete Guide with Math

> **RAG without Embeddings, without Vector Databases, without API costs.**

---

## What Problem Does RAG Solve?

LLMs like Claude or GPT are trained on **public internet data**.  
They know nothing about:
- Your company's internal documents
- Your PDF notes
- Your private knowledge base

**RAG (Retrieval Augmented Generation)** fixes this by:
1. Searching your documents for relevant content
2. Injecting that content into the LLM prompt as context
3. The LLM answers based on *your* data — not hallucinations

---

## Traditional RAG vs Vectorless RAG

| | Traditional Vector RAG | Vectorless RAG |
|---|---|---|
| **Retrieval** | Cosine similarity on embeddings | BM25 keyword scoring |
| **Embedding Model** | Required (OpenAI, etc.) | NOT needed |
| **Vector DB** | Required (Pinecone, Chroma) | NOT needed |
| **Cost** | Embedding API cost | Zero |
| **Setup** | Heavy | Minimal |
| **Semantic Search** | Yes | Partial |
| **Exact Keyword Match** | Can miss | Excellent |
| **Best For** | Large corpus, semantic queries | Small-medium, offline, keyword queries |

---

## How Does Vectorless RAG Work?

```
Your Documents
      |
      v
  BM25 Index  <-- just math on word frequencies, no neural network
      |
  User Query
      |
      v
  BM25 Score each document
      |
      v
  Top-K most relevant docs
      |
      v
  Build Prompt:  "Answer using these docs: ..."
      |
      v
  LLM (Claude / GPT / any model)
      |
      v
  Final Answer (grounded in your data)
```

**BM25 = Best Match 25** — the same algorithm used by Elasticsearch, Apache Solr, and most search engines. It has been battle-tested since the 1990s.

---

## Step 1: Knowledge Base — Your Private Documents

This is the data your RAG will search through.  
In a real project this could be: PDF chunks, database rows, wiki pages, etc.

We define a simple list of dictionaries — each with an `id`, `title`, and `content` field.

In [ ]:
DOCUMENTS = [
    {
        "id": 1,
        "title": "Python Basics",
        "content": "Python is a high-level programming language known for simplicity. "
                   "It supports multiple paradigms like OOP and functional programming. "
                   "Python uses indentation for code blocks instead of braces.",
    },
    {
        "id": 2,
        "title": "Machine Learning Overview",
        "content": "Machine learning is a subset of AI where models learn patterns from data. "
                   "Common algorithms include linear regression, decision trees, and neural networks. "
                   "Training requires labeled data in supervised learning.",
    },
    {
        "id": 3,
        "title": "RAG Systems",
        "content": "Retrieval Augmented Generation (RAG) combines retrieval with LLMs. "
                   "It fetches relevant documents and passes them as context to the language model. "
                   "RAG reduces hallucinations by grounding answers in real documents.",
    },
    {
        "id": 4,
        "title": "BM25 Algorithm",
        "content": "BM25 is a ranking function used in information retrieval. "
                   "It scores documents based on term frequency and inverse document frequency. "
                   "BM25 is used by Elasticsearch and many search engines. "
                   "It does not require embeddings or neural networks.",
    },
    {
        "id": 5,
        "title": "Vector Databases",
        "content": "Vector databases store high-dimensional embeddings for semantic search. "
                   "Examples include Pinecone, ChromaDB, Weaviate, and Qdrant. "
                   "They enable similarity search using cosine distance or dot product.",
    },
    {
        "id": 6,
        "title": "India Climate",
        "content": "India has diverse climate zones from tropical in the south to alpine in the north. "
                   "Monsoon season brings heavy rainfall from June to September. "
                   "Rajasthan has hot desert climate while Kerala has humid tropical weather.",
    },
    {
        "id": 7,
        "title": "Django Web Framework",
        "content": "Django is a Python web framework that follows the MVT pattern. "
                   "It includes ORM, authentication, admin panel out of the box. "
                   "Django is used for building scalable web applications quickly.",
    },
]

print(f"Loaded {len(DOCUMENTS)} documents into the knowledge base.")
for doc in DOCUMENTS:
    print(f"  [{doc['id']}] {doc['title']}")

---

## Step 2: The BM25 Math — Explained Simply

BM25 scores how relevant a document is to a query using two core ideas:

### TF — Term Frequency
> "If the query word 'BM25' appears 5 times in a document, that document is probably about BM25."

But raw count is not enough. A 1000-word document with 'BM25' appearing 5 times is less focused than a 20-word document with it 5 times.  
So we **normalize by document length**.

### IDF — Inverse Document Frequency
> "The word 'the' appears in every document — it is useless for ranking.  
> The word 'BM25' appears in only 1 document — it is very useful."

```
IDF(term) = log( (N - df + 0.5) / (df + 0.5) + 1 )

  N   = total number of documents
  df  = number of documents containing the term
```

### Final BM25 Formula

```
score(query, doc) = Sum over each term in query:

    IDF(term)  x  tf * (k1 + 1)
                  ----------------------------------------
                  tf + k1 * (1 - b + b * doc_len / avg_len)

Parameters:
  k1 = 1.5   controls term saturation (how much repeated words matter)
  b  = 0.75  controls document length normalization
             1.0 = full normalization, 0.0 = no normalization
```

Let's implement BM25 from scratch — **no libraries needed**.

In [ ]:
import math
import re
from collections import Counter
from typing import List, Dict


def tokenize(text: str) -> List[str]:
    """Lowercase and extract words only (removes punctuation and numbers)."""
    return re.findall(r'\b[a-z]+\b', text.lower())


class BM25Retriever:
    """
    BM25 retriever built from scratch.

    Steps during initialization:
      1. Tokenize all documents (title counted twice for a simple title boost)
      2. Compute average document length
      3. Build an inverted index: term -> list of (doc_index, term_frequency)
      4. Compute document frequency (df) for each term
    """

    def __init__(self, documents: List[Dict], k1: float = 1.5, b: float = 0.75):
        self.documents = documents
        self.k1 = k1
        self.b = b

        # Tokenize — title counted twice gives a mild title-match boost
        self.tokenized_docs = []
        for doc in documents:
            text = doc["title"] + " " + doc["title"] + " " + doc["content"]
            self.tokenized_docs.append(tokenize(text))

        self.N = len(self.tokenized_docs)
        self.avg_doc_len = sum(len(d) for d in self.tokenized_docs) / self.N

        # Inverted index: term -> [(doc_idx, term_freq), ...]
        self.inverted_index = {}
        for idx, tokens in enumerate(self.tokenized_docs):
            for term, freq in Counter(tokens).items():
                self.inverted_index.setdefault(term, []).append((idx, freq))

        # df[term] = number of documents that contain this term
        self.df = {term: len(postings) for term, postings in self.inverted_index.items()}

    def idf(self, term: str) -> float:
        """Compute IDF for a single term. Rare terms get higher IDF."""
        df = self.df.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query_tokens: List[str], doc_idx: int) -> float:
        """Compute BM25 score for one (query, document) pair."""
        doc_tokens = self.tokenized_docs[doc_idx]
        doc_len = len(doc_tokens)
        tf_map = Counter(doc_tokens)

        total = 0.0
        for term in query_tokens:
            tf = tf_map.get(term, 0)
            if tf == 0:
                continue  # term not in this document, contributes 0
            idf = self.idf(term)
            numerator   = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avg_doc_len)
            total += idf * (numerator / denominator)
        return total

    def retrieve(self, query: str, top_k: int = 3) -> List[Dict]:
        """Score all documents against the query and return top-k."""
        query_tokens = tokenize(query)
        scores = [(idx, self.score(query_tokens, idx)) for idx in range(self.N)]
        scores = [(idx, s) for idx, s in scores if s > 0]  # discard zero-score docs
        scores.sort(key=lambda x: x[1], reverse=True)

        results = []
        for idx, bm25_score in scores[:top_k]:
            doc = self.documents[idx].copy()
            doc["bm25_score"] = round(bm25_score, 4)
            results.append(doc)
        return results


# Build the retriever
retriever = BM25Retriever(DOCUMENTS)

print("BM25 Retriever built successfully!")
print(f"  Total documents indexed : {retriever.N}")
print(f"  Unique terms in index   : {len(retriever.inverted_index)}")
print(f"  Avg document length     : {retriever.avg_doc_len:.1f} tokens")

---

## Step 3: Watch BM25 Score Each Document

Let's pick one query and see the raw score for every document.  
This shows exactly **why** BM25 ranks documents the way it does.

Documents that share more query terms — especially rare ones — will score higher.

In [ ]:
query = "BM25 ranking algorithm for search"
query_tokens = tokenize(query)

print(f"Query        : '{query}'")
print(f"Query tokens : {query_tokens}")
print()
print(f"{'Rank':<5} {'BM25 Score':>12}  Document Title")
print("-" * 55)

all_scores = []
for idx in range(retriever.N):
    s = retriever.score(query_tokens, idx)
    all_scores.append((idx, s))

all_scores.sort(key=lambda x: x[1], reverse=True)

for rank, (idx, s) in enumerate(all_scores, 1):
    bar = "#" * int(s * 3)   # visual bar proportional to score
    print(f"  {rank:<3} {s:>10.4f}   {DOCUMENTS[idx]['title']:<30}  {bar}")

---

## Step 4: Term-by-Term BM25 Breakdown

Let's open the black box.  
We'll see **exactly how each query term contributes** to the final score for the top-ranked document.

Key things to notice:
- **Rare terms** (high IDF) contribute more to the score than common terms
- Terms that appear **more than once** get a higher TF contribution, but with diminishing returns
- Terms that **do not appear** in the document contribute 0

In [ ]:
def bm25_breakdown(retriever, query: str, doc_idx: int):
    """Print a per-term BM25 contribution table for a specific document."""
    query_tokens = tokenize(query)
    doc = retriever.documents[doc_idx]
    doc_tokens = retriever.tokenized_docs[doc_idx]
    doc_len = len(doc_tokens)
    tf_map = Counter(doc_tokens)

    print(f"Query    : '{query}'")
    print(f"Document : '{doc['title']}'")
    print(f"Doc length: {doc_len} tokens  |  Avg doc length: {retriever.avg_doc_len:.1f}")
    print()
    print(f"  {'Term':<15} {'TF':>4}  {'IDF':>7}  {'Score':>10}  Note")
    print("  " + "-" * 60)

    total = 0.0
    for term in query_tokens:
        tf = tf_map.get(term, 0)
        idf = retriever.idf(term)
        if tf > 0:
            num = tf * (retriever.k1 + 1)
            den = tf + retriever.k1 * (1 - retriever.b + retriever.b * doc_len / retriever.avg_doc_len)
            score = idf * (num / den)
            note = "matched"
        else:
            score = 0.0
            note = "NOT in doc"
        total += score
        print(f"  {term:<15} {tf:>4}  {idf:>7.4f}  {score:>10.4f}  {note}")

    print("  " + "-" * 60)
    print(f"  {'TOTAL':>38}  {total:>10.4f}")
    print()
    print("  IDF logic: rare terms (few docs contain them) get a HIGH IDF.")
    print("  TF  logic: more occurrences = higher score, but with diminishing returns.")


# Show the breakdown for the top-ranked document from Step 3
top_doc_idx = all_scores[0][0]
bm25_breakdown(retriever, "BM25 ranking algorithm for search", top_doc_idx)

---

## Step 5: Full RAG Pipeline

Now we connect everything into a complete pipeline:

```
Query  -->  BM25 Retrieve  -->  Build Prompt  -->  LLM  -->  Answer
```

- **Retrieval**: BM25 finds the most relevant documents for the query
- **Prompt Building**: Retrieved documents are injected into the LLM prompt as context
- **LLM**: Generates an answer grounded in the retrieved documents (not hallucinations)

In [ ]:
def build_prompt(query: str, retrieved_docs: list) -> str:
    """Combine retrieved documents into a prompt for the LLM."""
    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        context_parts.append(
            f"[Document {i}: {doc['title']} | BM25 Score: {doc['bm25_score']}]\n"
            f"{doc['content']}"
        )
    context = "\n\n".join(context_parts)

    return f"""You are a helpful assistant. Answer the question using ONLY the provided documents.
If the answer is not in the documents, say "I don't have information about that."

--- RETRIEVED DOCUMENTS ---
{context}
--- END DOCUMENTS ---

Question: {query}

Answer:"""


def vectorless_rag(query: str, top_k: int = 3, use_llm: bool = False) -> dict:
    """
    Full Vectorless RAG pipeline.

    Steps:
      1. BM25 retrieval  (no embeddings, no vector DB)
      2. Build LLM prompt with retrieved context
      3. Call LLM (or simulate if use_llm=False)
    """
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")

    # Step 1: Retrieve
    retrieved_docs = retriever.retrieve(query, top_k=top_k)
    print(f"\n[RETRIEVAL] Top {len(retrieved_docs)} documents:")
    for doc in retrieved_docs:
        print(f"  Score {doc['bm25_score']:>7}  -->  {doc['title']}")

    # Step 2: Build Prompt
    prompt = build_prompt(query, retrieved_docs)
    print(f"\n[PROMPT] (showing first 400 chars)")
    print(prompt[:400] + "\n...")

    # Step 3: Generate Answer
    if use_llm:
        answer = call_claude(prompt)
    else:
        # Demo mode: return the first sentence from the top document
        if retrieved_docs:
            top = retrieved_docs[0]
            answer = f"[DEMO] From '{top['title']}': " + top["content"].split(".")[0] + "."
        else:
            answer = "I don't have information about that in my knowledge base."

    print(f"\n[ANSWER]\n{answer}")
    return {"query": query, "retrieved": retrieved_docs, "prompt": prompt, "answer": answer}


print("RAG pipeline ready. Run the next cell to test it.")

---

## Step 6: Test the RAG — Demo Mode (No API Key Needed)

Run different queries and observe:
- Which documents get retrieved
- Their BM25 scores
- What a demo answer looks like

The last query (`"quantum computing"`) is **not in the knowledge base** — notice what happens.

In [ ]:
test_queries = [
    "What is BM25 and how does it work?",
    "Tell me about machine learning algorithms",
    "What is RAG and why is it used?",
    "How does Django framework work?",
    "What is the weather like in India?",
    "What is quantum computing?",   # NOT in the knowledge base
]

results = []
for q in test_queries:
    r = vectorless_rag(q, top_k=2, use_llm=False)
    results.append(r)
    print()

---

## Step 7: Connect to Claude API (Real LLM Answers)

Set your API key below to get real answers from Claude.  
Without a key, it falls back to demo mode automatically.

Get your key from: [console.anthropic.com](https://console.anthropic.com)

**Two ways to set the key:**
```bash
# Windows
set ANTHROPIC_API_KEY=sk-ant-...

# Mac / Linux
export ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
import os

# Reads from environment variable — do NOT paste keys directly into notebooks
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")


def call_claude(prompt: str) -> str:
    """Call Claude API with the RAG prompt and return the answer."""
    if not ANTHROPIC_API_KEY:
        return "[No API key set — running in demo mode. Set ANTHROPIC_API_KEY to enable real answers.]"
    try:
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        message = client.messages.create(
            model="claude-haiku-4-5-20251001",   # fast and cheap
            max_tokens=512,
            messages=[{"role": "user", "content": prompt}]
        )
        return message.content[0].text
    except ImportError:
        return "[ERROR] Run: pip install anthropic"
    except Exception as e:
        return f"[ERROR] {e}"


print("Claude function ready.")
print(f"API key set: {'YES' if ANTHROPIC_API_KEY else 'NO — demo mode will be used'}")

In [ ]:
# Run one query with real LLM (falls back to demo mode if no API key)
vectorless_rag(
    query="What is BM25 and how is it different from vector search?",
    top_k=3,
    use_llm=True
)

---

## Step 8: Add Your Own Documents

Replace or extend the `DOCUMENTS` list with your own content.  
That's it — **no re-training, no embedding, no vector DB migration**.

Just rebuild the `BM25Retriever` with the new list and you're ready to query.

In [ ]:
# Add your own documents to the existing knowledge base
my_docs = DOCUMENTS + [
    {
        "id": 8,
        "title": "FastAPI Framework",
        "content": "FastAPI is a modern Python web framework for building APIs quickly. "
                   "It uses Python type hints for automatic validation and OpenAPI docs. "
                   "FastAPI is async by default and very fast compared to Flask or Django.",
    },
    {
        "id": 9,
        "title": "Docker Containers",
        "content": "Docker packages applications into containers for consistent deployment. "
                   "A Dockerfile defines the environment and dependencies. "
                   "Docker Compose orchestrates multiple containers together.",
    },
]

# Rebuild the retriever with the expanded knowledge base
my_retriever = BM25Retriever(my_docs)
print(f"Expanded knowledge base: {my_retriever.N} documents")

# Test with a query targeting the new documents
results = my_retriever.retrieve("FastAPI async Python API", top_k=3)
print("\nQuery: 'FastAPI async Python API'")
for doc in results:
    print(f"  [{doc['bm25_score']}] {doc['title']}")

---

## Bonus: BM25 Math Walkthrough — Manual Calculation

Let's manually compute the BM25 score for one query-document pair, term by term.  
This completely opens the black box — you will see exactly what the numbers mean.

After running this cell, you should be able to trace any BM25 score by hand.

In [ ]:
import math
import re
from collections import Counter

query    = "BM25 ranking algorithm"
doc_text = ("BM25 is a ranking function used in information retrieval. "
            "It scores documents based on term frequency and inverse document frequency.")

print("BM25 MATH WALKTHROUGH")
print("=" * 60)
print(f"\nQuery   : '{query}'")
print(f"Document: '{doc_text[:60]}...'")

# Tokenize
query_tokens = re.findall(r'[a-z]+', query.lower())
doc_tokens   = re.findall(r'[a-z]+', doc_text.lower())
tf_map = Counter(doc_tokens)

# BM25 parameters
N           = 7      # total documents in the corpus
df_term     = 1      # number of docs that contain 'bm25'
doc_len     = len(doc_tokens)
avg_doc_len = 35     # approximate average document length
k1, b       = 1.5, 0.75

print(f"\nQuery tokens : {query_tokens}")
print(f"\n{'Term':<15} {'TF':>4} {'IDF':>7} {'BM25 Score':>12}")
print("-" * 42)

total = 0
for term in query_tokens:
    tf  = tf_map.get(term, 0)
    idf = math.log((N - df_term + 0.5) / (df_term + 0.5) + 1) if tf > 0 else 0
    if tf > 0:
        num   = tf * (k1 + 1)
        denom = tf + k1 * (1 - b + b * doc_len / avg_doc_len)
        score = idf * num / denom
    else:
        score = 0
    total += score
    print(f"{term:<15} {tf:>4} {idf:>7.3f} {score:>12.4f}")

print("-" * 42)
print(f"{'TOTAL':>30} {total:>10.4f}")
print("\n=> Higher total score = more relevant document")

---

## Summary

```
Vectorless RAG in 3 steps:

  1. STORE   -- Put your documents in a plain Python list
  2. SEARCH  -- BM25 scores each document using keyword math (no embeddings)
  3. ANSWER  -- Top-K documents go into the LLM prompt as context
```

---

### When to use Vectorless RAG
- Small to medium knowledge base (fewer than ~100k documents)
- No embedding API budget
- Offline or edge device deployment
- Exact keyword matching matters (legal, medical, code search)
- You want a simple, fully debuggable system

### When to upgrade to Vector RAG
- Queries rely on meaning, not exact words (e.g., "Find documents about sadness")
- Multilingual queries
- Millions of documents
- Semantic similarity matters more than keyword overlap

---

### Production Libraries (when you outgrow this notebook)

| Library | Description |
|---|---|
| `rank_bm25` | Pure Python BM25 (same algorithm, optimized) |
| `whoosh` | Full-text search engine in Python |
| `sqlite FTS5` | BM25 built into SQLite — zero dependencies |
| `Elasticsearch` | BM25 at scale with a rich query DSL |

```python
# Using rank_bm25 library (pip install rank-bm25)
from rank_bm25 import BM25Okapi

corpus = [doc["content"].split() for doc in DOCUMENTS]
bm25   = BM25Okapi(corpus)
scores = bm25.get_scores("BM25 ranking".split())
```